# CASMI26: real scores, tie blocks, and what the top public notebooks are made of

The public notebooks in this competition fork each other quickly: fusion pipelines get re-weighted,
re-titled and re-published within hours. Before spending a GPU run on a fork, three things are worth
checking, and this notebook checks all of them live from Kaggle (turn **Internet on**):

1. **The shape of the leaderboard.** Scores are shown to three decimals, so many teams share the same
   displayed score. Inside such a block, rank follows the unrounded score.
2. **The real best score of every public notebook**, set against the score its title claims.
3. **What the top notebooks attach**, and which ones depend on an input a copy cannot load.

Nothing here reads competition data.

In [ ]:
import re
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import pandas as pd
import requests

COMPETITION = "enveda-CASMI26-molecule-id-mass-spectra"
pd.set_option("display.max_colwidth", 70)
_s = requests.Session()
_s.get("https://www.kaggle.com/competitions", timeout=60)
_h = {"x-xsrf-token": _s.cookies.get("XSRF-TOKEN", "")}


def kaggle_api(service, method, body, tries=4):
    for attempt in range(tries):
        r = _s.post(f"https://www.kaggle.com/api/i/{service}/{method}", json=body, headers=_h, timeout=90)
        if r.status_code == 200:
            return r.json()
        time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"{method}: HTTP {r.status_code}. Is Internet turned on?")


comp = kaggle_api("competitions.CompetitionService", "GetCompetition", {"competitionName": COMPETITION})
comp = comp.get("competition", comp)
n_teams = int(comp["totalTeams"])
board = pd.DataFrame(kaggle_api("competitions.LeaderboardService", "GetLeaderboard", {"competitionId": comp["id"]})["publicLeaderboard"])
board["score"] = board["displayScore"].astype(float)
cuts = {"gold": 10 + n_teams // 500, "silver": n_teams // 20, "bronze": n_teams // 10}
print(f"{comp['title']}: {n_teams:,} teams, deadline {comp['deadline'][:10]}")

## 1. The leaderboard's tie blocks

How many teams share each displayed score near the top, and where today's medal lines fall. A copied
notebook lands inside its score's block; to move up you need a real gain, even a small one.

In [ ]:
blocks = (board.groupby("score").agg(teams=("rank", "size"), best_rank=("rank", "min"), worst_rank=("rank", "max"))
          .sort_index(ascending=False).reset_index())
blocks["medal line inside"] = [" + ".join(m for m, r in cuts.items() if lo <= r <= hi)
                               for lo, hi in zip(blocks.best_rank, blocks.worst_rank)]
top = blocks.head(18)
INK, INK2, MUTED, GRID, BLUE, ORANGE, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#2a78d6", "#eb6834", "#fcfcfb"
fig, ax = plt.subplots(figsize=(11, 6.2), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.barh([f"{s:.3f}" for s in top["score"]], top["teams"], color=BLUE, height=0.72)
ax.invert_yaxis()
for y, (t, lo, hi, cut) in enumerate(zip(top.teams, top.best_rank, top.worst_rank, top["medal line inside"])):
    ax.text(t + top.teams.max() * 0.01, y, f"{t:,} teams, ranks {lo:,}-{hi:,}" + (f"   <- {cut} line inside" if cut else ""),
            va="center", fontsize=9.5, color=INK2)
ax.set_xlabel("teams with exactly this displayed score", color=INK2); ax.set_ylabel("public score", color=INK2)
ax.set_title(f"How many teams share each score ({n_teams:,} teams)", loc="left", fontsize=13, color=INK, fontweight="bold")
ax.grid(axis="x", color=GRID, linewidth=0.8); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
for s in ("left", "bottom"): ax.spines[s].set_color(MUTED)
ax.tick_params(colors=INK2)
ax.set_xlim(0, top.teams.max() * 1.55)
plt.tight_layout(); plt.show()
top

## 2. Real scores of the public notebooks, against their titles

In [ ]:
raw = {}
for sort_by in ("SCORE_DESCENDING", "VOTE_COUNT", "DATE_RUN"):
    for page in range(1, 8):
        body = {"kernelFilterCriteria": {"search": "", "listRequest": {"competitionId": comp["id"], "sortBy": sort_by,
                                                                       "pageSize": 100, "group": "EVERYONE", "page": page}},
                "detailFilterCriteria": {"deletedAccessBehavior": "RETURN_NOTHING", "unauthorizedAccessBehavior": "RETURN_NOTHING",
                                         "excludeResultsFilesOutputs": True}, "readMask": "pinnedKernels"}
        got = kaggle_api("kernels.KernelsService", "ListKernels", body).get("kernels", [])
        if not got:
            break
        for k in got:
            raw[k["scriptUrl"]] = k
nbs = pd.DataFrame([{"notebook": k["scriptUrl"].removeprefix("/code/"), "title": k.get("title", ""),
                     "real score": k.get("bestPublicScore"), "votes": k.get("totalVotes") or 0,
                     "updated": str(k.get("scriptVersionDateCreated", ""))[:10],
                     "inputs": sorted(d.get("dataSourceUrl", "").split("/", 2)[-1] for d in (k.get("dataSources") or [])
                                      if d.get("dataSourceUrl", "").startswith(("/datasets/", "/models/", "/code/"))),
                     "hidden inputs": sum(1 for d in (k.get("dataSources") or []) if not d.get("dataSourceUrl"))}
                    for k in raw.values()])


def claimed(row):
    text = row["title"] + " " + re.sub(r"(?<=\d)-(?=\d)", ".", row["notebook"])
    found = [float(x) for x in re.findall(r"\b0\.\d{3,4}\b", text)]
    return max(found) if found else None


nbs["claimed"] = nbs.apply(claimed, axis=1)
scored = nbs.dropna(subset=["real score"])
claims = scored.dropna(subset=["claimed"]).copy()
claims["claim minus real"] = (claims["claimed"] - claims["real score"]).round(3)
print(f"{len(nbs):,} public notebooks, {len(scored):,} with a recorded score, {len(claims)} claim a score in the title or URL; "
      f"{(claims['claim minus real'] >= 0.002).sum()} claim at least 0.002 more than they ever scored")
claims.sort_values("claim minus real", ascending=False)[["notebook", "claimed", "real score", "claim minus real", "votes"]].head(12).reset_index(drop=True)

## 3. The best public notebooks, and what they attach

Sorted by real score. `hidden inputs` counts attached inputs that do not appear on the public page,
usually the author's private files: a copy runs without them and scores lower.

In [ ]:
best = scored.sort_values(["real score", "votes"], ascending=False).head(25)
show = best[["notebook", "real score", "votes", "updated", "hidden inputs"]].reset_index(drop=True)
show.index += 1
use, top_score = Counter(), defaultdict(float)
for inputs, s in zip(best["inputs"], best["real score"]):
    for d in set(inputs):
        use[d] += 1; top_score[d] = max(top_score[d], s)
print("datasets attached by the 25 best public notebooks:")
for d, n in use.most_common(15):
    print(f"   {n:2d} notebooks, best {top_score[d]:.3f}  {d}")
show

## Takeaways

- **Most displayed scores near the top are shared by many teams.** A plain fork lands at the back of its
  block; a small real improvement moves you past part of it.
- **Most top notebooks draw on the same few public datasets** (section 3). Re-weighting the same models
  rarely buys more than a tie.
- **Check `hidden inputs` and the title claim before forking.** Both are listed above.

If this saved you a run, an upvote helps others find it. Corrections welcome in the comments.